# 项目启动与数据接入
运营商给了我们 沈阳 5,644 个手机用户在 2018 年 10 月 3 日（国庆第三天）一整天的 53 万条信令（已脱敏）。 我们需要做完一个完整的数据项目：先把数据验收清楚，再识别停留和出行、定义指标口径、得出结论， 最后把这套分析改造成「每天能自动运行、坏数据进不来、每个数字能追溯、能对外展示」的系统。

## 手上有什么数据

全部装在 `data/signal.db` 里（SQLite 单文件库）。**学生拿到的就是这个库，没有原始 CSV**——企业里分析师面对的是数仓里的表，原始文件留在数据平台，这和真实工作一致。
`ods_` = 原始层，**一个值都没改**，入库时只做了起列名、补顶点序号这类不改变含义的事。

| 表 | 行数 | 是什么 |
|---|---|---|
| `ods_signal` | 532,828 | 信令：谁（IMSI）在什么时刻连了哪个基站扇区 |
| `ods_cell` | 49,954 | 基站扇区的经纬度——信令里只有编号，位置全靠它补 |
| `ods_poi_purpose` | 1,273 | 兴趣点：居住、商业、办公…… |
| `ods_poi_transport` | 968 | 地铁站、公交站 |
| `ods_aoi_v1 / v2 / json` | 104 / 1,685 / 1,716 | 商圈边界的**三个版本**，D3 要甄别该信哪个 |
| `ods_district_boundary` | 2,951 | 沈阳各区县的边界顶点 |
| `ods_nid_mapping` | 24 | `nid` 字段的字典（D2 选讲里要验证它可不可信） |

### 数据在项目当中如何流动

![数据流动](iamge/数据流动.png)

### 项目需求
- 客户需求：国庆期间想要了解一下区里的夜间经济情况，运行商有数据，你帮我们看看

一个能回答的问题至少要能说清四样东西：看那个对象，看什么指标，看哪段时间，跟谁比

### 环境准备
第一步：先保证环境是OK的

In [ ]:
import sqlite3  # 读取sqlite数据库的
from pathlib import Path # 现代化处理路径的

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent  # notebook放在notebook文件里面上一级是根目录
DB_PATH = ROOT / "data" / "signal.db"  # 数据库文件
IMG_DIR = ROOT / "image"
IMG_DIR.mkdir(exist_ok = True)

assert DB_PATH.exists(), f"找不到数据库文件{DB_PATH}"

FONTS = ['TeX Gyre Termes', "Noto Serif CJK SC"]
plt.rcParams.update({
    "font.family": FONTS, "axes.unicode_minus": True,
    "axes.spines.top": False, "axes.spines.right": False, 
    "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "y",
    "axes.axisbelow": True, "grid.color": "#E5E7EB",
    "axes.edgecolor": "#9CA3AF",
    "xtick.color": "#4B5563", "ytick.color": "#4B5563",
    "xtick.major.size": 0, "ytick.major.size": 0,
    "legend.frameon": False,
    "figure.dpi": 100, "savefig.dpi": 300,
})
# 配色：灰色打底，一种颜色强调一件事，红色只表示警示
MAIN, MAIN_LIGHT, GRAY, RED = "#1D6FA5", "#DCEAF5", "#C4CAD3", "#D64545"
INK, MUTED, FAINT = "#1F2937", "#6B7280", "#9CA3AF"
MAIN_DARK, MAIN_MID, GRAY_DARK, AMBER = "#174E73", "#7FB3D9", "#8E99A6", "#B7791F"
AREA_COLOR = {"西塔": MAIN, "太原街商圈": "#D9822B",
              "中街商圈": "#3B8C66",          # 同一区域在所有图里同色
              "沈阳北站": "#7A5EA8", "沈阳世博园":
              "#B04A6A", "棋盘山风景区": "#6B7F2A"}
SEG_COLOR = {"市内来访": MAIN, "周边居民": MAIN_MID,
             "外地来客": "#D9822B", "市外来访": "#D9822B",
             "过夜地不明": GRAY}      # “不明”永远是灰色：不是一类人，是不知道
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
con = sqlite3.connect(DB_PATH)    # 整个 notebook 共用一个数据库连接

In [ ]:
# 自检：版本、中文字体、数据库大小
import sys, matplotlib, seaborn
from matplotlib import font_manager

installed = {f.name for f in font_manager.fontManager.ttflist}
cn_font = list(f for f in plt.rcParams["font.family"] 
            if f in installed and f != "DejaVu Sans")

print(f"Python      {sys.version.split()[0]}")
print(f"pandas      {pd.__version__}")
print(f"numpy       {np.__version__}")
print(f"matplotlib  {matplotlib.__version__}")
print(f"seaborn     {seaborn.__version__}")
print(f"SQLite      {sqlite3.sqlite_version}")
print(f"中文字体    {cn_font}")
print(f"数据库      {DB_PATH.name}（{DB_PATH.stat().st_size / 1e6:.0f} MB）")

In [ ]:
# 连接数据库读取数据到pandas

tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name", con)
tables['行数'] = [
    pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", con)["n"][0]
    for t in tables["name"]
]

raw = pd.read_sql("SELECT * FROM ods_signal", con)
raw.head()